## **Revision audit 01 -- Patient overlap between training and evaluation samples (submitted version)**

**Purpose.** Quantify, for the *submitted* analyses, how many patients (and event-bearing patients) appear on both
sides of the evaluation boundary. This is the evidence requested in reviewer comments **R1-4** and **R2-2**
(plan task **T43**). It does **not** change any model, label or dataset.

**What it does**

1. Loads only `subject_id`, `hadm_id` and the *submitted* outcome label from the two original analysis tables
   (`cohort_maternal_dataset` -> Model A, `cohort_smm_dataset` -> Model B).
2. Describes admissions per patient.
3. **Replicates** the submitted partitions exactly as the original notebooks built them
   (`sort_values('subject_id')`, `train_test_split(test_size=0.2, random_state=42, stratify=y)`, and
   `StratifiedKFold(5, shuffle=True, random_state=42)`), and counts the overlap.
4. **Simulates** 1,000 alternative random stratified hold-out splits (and 200 repetitions of 5-fold CV), to give the
   overlap an order-independent range. This matters because the exact replication depends on the physical row order
   returned by PostgreSQL; the checks in section 3 tell you whether the replication reproduced the submitted split sizes.

**Data-use rules.** Only aggregate counts are printed. No identifier is displayed, exported or saved. The notebook
writes nothing to disk. Do **not** add cells that print or export rows.

**Run order.** Run all cells top to bottom (needs the local PostgreSQL connection, as in the other notebooks).
Then paste back the output of the **last cell** (section 7) only.

### **1. Libraries and connection**

In [1]:
import os, sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

# notebooks/revision/ -> project root is two levels up
sys.path.append(os.path.abspath(os.path.join("..", "..")))
import libraries.database as db_lib
engine = db_lib.get_connect_database()

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

### **2. Load identifiers and the submitted labels**

Only three columns are read. The two tables are the ones the submitted notebooks loaded
(`db_lib.get_maternal_cohort_query()` and `db_lib.get_smm_cohort_query()`); selecting fewer columns does not change the
row order of a sequential scan. The expected positive counts (41 and 475) confirm these are the submitted labels.

In [2]:
SPECS = {
    "A": dict(name="Model A - mortality (submitted label)",
              query="SELECT subject_id, hadm_id, outcome FROM mimiciv.cohort_maternal_dataset;",
              expected_pos=41, expected_train=(20588, 33), expected_test=(5148, 8)),
    "B": dict(name="Model B - SMM (submitted label)",
              query="SELECT subject_id, hadm_id, outcome_smm AS outcome FROM mimiciv.cohort_smm_dataset;",
              expected_pos=475, expected_train=(20588, 380), expected_test=(5148, 95)),
}

raw = {}
for key, sp in SPECS.items():
    d = pd.read_sql(sp["query"], engine)
    d["outcome"] = d["outcome"].astype(int)
    raw[key] = d
    ok = "OK" if int(d["outcome"].sum()) == sp["expected_pos"] else "CHECK: differs from submitted count"
    print(f"{sp['name']}: {len(d)} admissions, {d['subject_id'].nunique()} patients, "
          f"{int(d['outcome'].sum())} positive admissions  [{ok}]")

Model A - mortality (submitted label): 25736 admissions, 15084 patients, 41 positive admissions  [OK]
Model B - SMM (submitted label): 25736 admissions, 15084 patients, 475 positive admissions  [OK]


### **3. Rebuild the submitted row order and check the replication**

The original notebooks did `df_model = df_scaled.sort_values('subject_id').reset_index(drop=True)` before splitting
(default sort kind, unstable for ties). The same call is applied here. The hold-out split depends only on the order of
the outcome vector, so if train/test sizes and positives match the submitted values the replication is consistent
(it cannot prove identical row-by-row membership; section 6 gives the order-independent range).

In [3]:
def ordered(d):
    """Same ordering step as the submitted notebooks (Model A cell 46, Model B cell 47)."""
    return d.sort_values("subject_id").reset_index(drop=True)

data = {}
for key, d in raw.items():
    o = ordered(d)
    codes, _ = pd.factorize(o["subject_id"])          # integer patient codes; identifiers are not kept or shown
    data[key] = dict(y=o["outcome"].to_numpy(), pat=codes, n_pat=int(codes.max()) + 1)

def holdout_idx(y, seed=42):
    idx = np.arange(len(y))
    tr, te = train_test_split(idx, test_size=0.2, random_state=seed, stratify=y)
    return tr, te

rows = []
for key, sp in SPECS.items():
    y = data[key]["y"]
    tr, te = holdout_idx(y)
    got_tr, got_te = (len(tr), int(y[tr].sum())), (len(te), int(y[te].sum()))
    rows.append({"model": key, "train (n, events) replicated": got_tr, "train submitted": sp["expected_train"],
                 "test (n, events) replicated": got_te, "test submitted": sp["expected_test"],
                 "match": got_tr == sp["expected_train"] and got_te == sp["expected_test"]})
pd.DataFrame(rows)

,model,"train (n, events) replicated",train submitted,"test (n, events) replicated",test submitted,match
0,A,"(20588, 33)","(20588, 33)","(5148, 8)","(5148, 8)",True
1,B,"(20588, 380)","(20588, 380)","(5148, 95)","(5148, 95)",True


### **4. Admissions per patient**

In [4]:
rows = []
for key, sp in SPECS.items():
    pat, y, P = data[key]["pat"], data[key]["y"], data[key]["n_pat"]
    per_pat = np.bincount(pat, minlength=P)
    ev_pat = np.unique(pat[y == 1])
    rows.append({
        "model": key,
        "admissions": len(y),
        "patients": P,
        "patients with >1 admission": int((per_pat > 1).sum()),
        "% patients with >1 admission": round(100 * (per_pat > 1).mean(), 1),
        "median admissions per patient": float(np.median(per_pat)),
        "max admissions per patient": int(per_pat.max()),
        "positive admissions": int(y.sum()),
        "event-bearing patients (unique)": int(len(ev_pat)),
        "positive admissions per event-bearing patient: max": int(np.bincount(pat[y == 1])[ev_pat].max()),
    })
display(pd.DataFrame(rows).set_index("model").T)

dist = {}
for key in SPECS:
    per_pat = np.bincount(data[key]["pat"], minlength=data[key]["n_pat"])
    dist[key] = pd.Series(np.minimum(per_pat, 5)).value_counts().sort_index()
dist = pd.DataFrame(dist).fillna(0).astype(int)
dist.index = [f"{i} admission(s)" if i < 5 else "5 or more" for i in dist.index]
display(dist)

model,A,B
admissions,25736.0,25736.0
patients,15084.0,15084.0
patients with >1 admission,6793.0,6793.0
% patients with >1 admission,45.0,45.0
median admissions per patient,1.0,1.0
max admissions per patient,27.0,27.0
positive admissions,41.0,475.0
event-bearing patients (unique),26.0,426.0
positive admissions per event-bearing patient: max,6.0,7.0


,A,B
1 admission(s),8291,8291
2 admission(s),4418,4418
3 admission(s),1558,1558
4 admission(s),491,491
5 or more,326,326


### **5. Overlap statistics for one partition**

For a given training set and evaluation set (indices into the ordered data):

* patients of the evaluation set that also appear in the training set;
* evaluation admissions whose patient also has an admission in the training set;
* positive evaluation admissions whose patient has any admission / a *positive* admission in the training set;
* event-bearing patients on both sides.

In [5]:
def overlap_stats(pat, y, P, tr, te):
    in_tr = np.zeros(P, dtype=bool); in_tr[pat[tr]] = True
    in_te = np.zeros(P, dtype=bool); in_te[pat[te]] = True
    pos_tr = np.zeros(P, dtype=bool); pos_tr[pat[tr][y[tr] == 1]] = True
    pos_te = np.zeros(P, dtype=bool); pos_te[pat[te][y[te] == 1]] = True
    te_pos_mask = y[te] == 1
    te_pat = pat[te]
    return {
        "train admissions": len(tr), "test admissions": len(te),
        "train events": int(y[tr].sum()), "test events": int(y[te].sum()),
        "train patients": int(in_tr.sum()), "test patients": int(in_te.sum()),
        "test patients also in train": int((in_te & in_tr).sum()),
        "% test patients also in train": 100 * (in_te & in_tr).sum() / max(in_te.sum(), 1),
        "test admissions with patient in train": int(in_tr[te_pat].sum()),
        "% test admissions with patient in train": 100 * in_tr[te_pat].mean(),
        "positive test admissions whose patient is in train": int(in_tr[te_pat[te_pos_mask]].sum()),
        "positive test admissions whose patient has a positive admission in train": int(pos_tr[te_pat[te_pos_mask]].sum()),
        "event-bearing patients in train": int(pos_tr.sum()),
        "event-bearing patients in test": int(pos_te.sum()),
        "event-bearing patients in both": int((pos_tr & pos_te).sum()),
    }

### **6. Submitted partitions (replicated) and order-independent simulation**

**6.1 Hold-out (80/20, seed 42).** **6.2 Five-fold stratified CV (shuffle, seed 42), counting each fold as the
evaluation set and the other four as training.** **6.3 Simulation:** 1,000 hold-out splits with seeds 0-999, and 200
repetitions of 5-fold CV; median and 2.5th-97.5th percentile.

In [6]:
# 6.1 hold-out, replicated
ho = {}
for key in SPECS:
    d = data[key]
    tr, te = holdout_idx(d["y"], seed=42)
    ho[key] = overlap_stats(d["pat"], d["y"], d["n_pat"], tr, te)
tab_ho = pd.DataFrame(ho)
tab_ho.columns = [f"Model {c}" for c in tab_ho.columns]
print("6.1 Hold-out 80/20 (replicated submitted split)")
display(tab_ho.round(1))

6.1 Hold-out 80/20 (replicated submitted split)


,Model A,Model B
train admissions,20588.0,20588.0
test admissions,5148.0,5148.0
train events,33.0,380.0
test events,8.0,95.0
train patients,13243.0,13196.0
test patients,4527.0,4523.0
test patients also in train,2686.0,2635.0
% test patients also in train,59.3,58.3
test admissions with patient in train,3086.0,3065.0
% test admissions with patient in train,59.9,59.5


In [7]:
# 6.2 five-fold CV, replicated (all folds shown, then the sum over folds)
cv_tabs = {}
for key in SPECS:
    d = data[key]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    X_dummy = np.zeros(len(d["y"]))
    fold_rows = []
    for k, (tr, te) in enumerate(skf.split(X_dummy, d["y"]), 1):
        s = overlap_stats(d["pat"], d["y"], d["n_pat"], tr, te)
        s["fold"] = k
        fold_rows.append(s)
    cv_tabs[key] = pd.DataFrame(fold_rows).set_index("fold")
    print(f"6.2 Model {key}: 5-fold CV, per fold")
    display(cv_tabs[key][["test admissions", "test events", "test patients", "test patients also in train",
                          "% test patients also in train", "% test admissions with patient in train",
                          "positive test admissions whose patient is in train",
                          "positive test admissions whose patient has a positive admission in train",
                          "event-bearing patients in both"]].round(1))

6.2 Model A: 5-fold CV, per fold


,test admissions,test events,test patients,test patients also in train,% test patients also in train,% test admissions with patient in train,positive test admissions whose patient is in train,positive test admissions whose patient has a positive admission in train,event-bearing patients in both
fold,,,,,,,,,
1,5148,9,4509,2670,59.2,59.9,6,6,4
2,5147,8,4555,2739,60.1,60.7,5,5,4
3,5147,8,4558,2699,59.2,60.2,4,4,4
4,5147,8,4537,2680,59.1,60.0,5,5,5
5,5147,8,4525,2649,58.5,59.4,3,3,3


6.2 Model B: 5-fold CV, per fold


,test admissions,test events,test patients,test patients also in train,% test patients also in train,% test admissions with patient in train,positive test admissions whose patient is in train,positive test admissions whose patient has a positive admission in train,event-bearing patients in both
fold,,,,,,,,,
1,5148,95,4553,2683,58.9,59.6,47,17,14
2,5147,95,4520,2723,60.2,61.3,49,12,12
3,5147,95,4542,2649,58.3,59.0,50,18,17
4,5147,95,4539,2746,60.5,61.9,53,10,9
5,5147,95,4562,2694,59.1,59.7,45,10,8


In [8]:
# 6.3 simulation (order-independent)
def summarize(rows):
    df = pd.DataFrame(rows)
    out = {}
    for c in df.columns:
        lo, med, hi = np.percentile(df[c], [2.5, 50, 97.5])
        out[c] = f"{med:.1f} ({lo:.1f}-{hi:.1f})"
    return pd.Series(out)

KEYS = ["% test patients also in train", "% test admissions with patient in train",
        "positive test admissions whose patient is in train",
        "positive test admissions whose patient has a positive admission in train",
        "event-bearing patients in both"]

sim_ho, sim_cv = {}, {}
for key in SPECS:
    d = data[key]
    rows = []
    for seed in range(1000):
        tr, te = holdout_idx(d["y"], seed=seed)
        s = overlap_stats(d["pat"], d["y"], d["n_pat"], tr, te)
        rows.append({k: s[k] for k in KEYS})
    sim_ho[f"Model {key}"] = summarize(rows)

    rows = []
    for rep in range(200):
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=rep)
        agg = {k: [] for k in KEYS}
        for tr, te in skf.split(np.zeros(len(d["y"])), d["y"]):
            s = overlap_stats(d["pat"], d["y"], d["n_pat"], tr, te)
            for k in KEYS:
                agg[k].append(s[k])
        # per repetition: mean of the percentages, sum of the counts over the 5 folds
        rows.append({k: (np.mean(v) if k.startswith("%") else np.sum(v)) for k, v in agg.items()})
    sim_cv[f"Model {key}"] = summarize(rows)

print("6.3a Hold-out: 1,000 random stratified 80/20 splits -- median (2.5th-97.5th percentile)")
display(pd.DataFrame(sim_ho))
print("6.3b 5-fold CV: 200 repetitions -- median (2.5th-97.5th percentile); % = mean over folds, counts = summed over folds")
display(pd.DataFrame(sim_cv))

6.3a Hold-out: 1,000 random stratified 80/20 splits -- median (2.5th-97.5th percentile)


,Model A,Model B
% test patients also in train,59.2 (58.0-60.4),59.2 (58.1-60.5)
% test admissions with patient in train,60.1 (58.9-61.3),60.1 (58.8-61.4)
positive test admissions whose patient is in train,4.0 (2.0-7.0),48.0 (39.0-57.0)
positive test admissions whose patient has a positive admission in train,4.0 (2.0-7.0),14.0 (8.0-20.0)
event-bearing patients in both,4.0 (1.0-6.0),12.0 (7.0-17.0)


6.3b 5-fold CV: 200 repetitions -- median (2.5th-97.5th percentile); % = mean over folds, counts = summed over folds


,Model A,Model B
% test patients also in train,59.2 (58.9-59.6),59.2 (58.9-59.6)
% test admissions with patient in train,60.1 (59.7-60.6),60.1 (59.7-60.5)
positive test admissions whose patient is in train,21.0 (18.0-23.0),240.0 (230.0-249.0)
positive test admissions whose patient has a positive admission in train,21.0 (18.0-23.0),72.0 (63.0-78.0)
event-bearing patients in both,18.0 (15.0-21.0),61.0 (53.0-68.1)


### **7. Compact summary -- paste this cell's output back**

One short block with the numbers that go into the response letter (R1-4 / R2-2). Aggregates only.

In [9]:
lines = []
for key, sp in SPECS.items():
    d = data[key]; P = d["n_pat"]
    per_pat = np.bincount(d["pat"], minlength=P)
    s = ho[key]
    cvt = cv_tabs[key]
    match = all([
        (s["train admissions"], s["train events"]) == sp["expected_train"],
        (s["test admissions"], s["test events"]) == sp["expected_test"],
    ])
    lines.append(f"[Model {key}] {sp['name']}")
    lines.append(f"  cohort: {len(d['y'])} adm, {P} patients, {int((per_pat > 1).sum())} with >1 adm "
                 f"({100 * (per_pat > 1).mean():.1f}%), positive adm {int(d['y'].sum())}, "
                 f"event-bearing patients {len(np.unique(d['pat'][d['y'] == 1]))}")
    lines.append(f"  hold-out replication matches submitted sizes/events: {match}")
    lines.append(f"  hold-out: test patients {s['test patients']}, also in train {s['test patients also in train']} "
                 f"({s['% test patients also in train']:.1f}%); test adm with patient in train "
                 f"{s['test admissions with patient in train']}/{s['test admissions']} "
                 f"({s['% test admissions with patient in train']:.1f}%)")
    lines.append(f"  hold-out events: test {s['test events']}; patient in train {s['positive test admissions whose patient is in train']}; "
                 f"patient with positive adm in train {s['positive test admissions whose patient has a positive admission in train']}; "
                 f"event-bearing patients train/test/both {s['event-bearing patients in train']}/"
                 f"{s['event-bearing patients in test']}/{s['event-bearing patients in both']}")
    lines.append(f"  5-fold CV (replicated): mean % test patients also in train {cvt['% test patients also in train'].mean():.1f}; "
                 f"mean % test adm with patient in train {cvt['% test admissions with patient in train'].mean():.1f}; "
                 f"positive test adm with patient in train (sum of folds) {int(cvt['positive test admissions whose patient is in train'].sum())}/"
                 f"{int(cvt['test events'].sum())}; with positive adm in train "
                 f"{int(cvt['positive test admissions whose patient has a positive admission in train'].sum())}")
    lines.append(f"  simulation hold-out (1000 splits): % test adm with patient in train {sim_ho['Model ' + key]['% test admissions with patient in train']}; "
                 f"events with patient in train {sim_ho['Model ' + key]['positive test admissions whose patient is in train']}")
print("\n".join(lines))

[Model A] Model A - mortality (submitted label)
  cohort: 25736 adm, 15084 patients, 6793 with >1 adm (45.0%), positive adm 41, event-bearing patients 26
  hold-out replication matches submitted sizes/events: True
  hold-out: test patients 4527, also in train 2686 (59.3%); test adm with patient in train 3086/5148 (59.9%)
  hold-out events: test 8; patient in train 6; patient with positive adm in train 6; event-bearing patients train/test/both 24/7/5
  5-fold CV (replicated): mean % test patients also in train 59.2; mean % test adm with patient in train 60.0; positive test adm with patient in train (sum of folds) 23/41; with positive adm in train 23
  simulation hold-out (1000 splits): % test adm with patient in train 60.1 (58.9-61.3); events with patient in train 4.0 (2.0-7.0)
[Model B] Model B - SMM (submitted label)
  cohort: 25736 adm, 15084 patients, 6793 with >1 adm (45.0%), positive adm 475, event-bearing patients 426
  hold-out replication matches submitted sizes/events: True
  